# Network Intrusion Detection 


+ negative sampling 
+ added degree to node features
+ added additional edge features



## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import NNConv

from sklearn.metrics import (roc_auc_score, average_precision_score,
                              precision_recall_curve, f1_score,
                              fbeta_score, precision_score, recall_score)
import matplotlib.pyplot as plt



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. Helper functions — parse GraphML

In [ ]:
def parse_graphml(file_path):
    """Parse GraphML file and return NetworkX graph."""
    try:
        G = nx.read_graphml(file_path)
        return G
    except Exception:
        return parse_graphml_manual(file_path)


def parse_graphml_manual(file_path):
    """Manual fallback parsing of GraphML."""
    tree = ET.parse(file_path)
    root = tree.getroot()
    G = nx.Graph()

    keys = {}
    for key_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}key'):
        keys[key_elem.get('id')] = key_elem.get('attr.name')

    for node_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}node'):
        G.add_node(node_elem.get('id'))

    for edge_elem in root.findall('.//{http://graphml.graphdrawing.org/xmlns}edge'):
        source = edge_elem.get('source')
        target = edge_elem.get('target')
        edge_attrs = {}
        for data_elem in edge_elem.findall('.//{http://graphml.graphdrawing.org/xmlns}data'):
            key_id = data_elem.get('key')
            attr_name = keys.get(key_id)
            if attr_name:
                try:
                    edge_attrs[attr_name] = float(data_elem.text)
                except Exception:
                    edge_attrs[attr_name] = data_elem.text
        G.add_edge(source, target, **edge_attrs)

    return G


def extract_edge_features(G):
    """Extract edge features as a pandas DataFrame."""
    rows = []
    for u, v, data in G.edges(data=True):
        row = {'source': u, 'target': v}
        row.update(data)
        rows.append(row)
    return pd.DataFrame(rows)

## 3. Load graph

In [ ]:
FILE_PATH = "data/0.1M-Stratified-Multi.graphml"

G_full = parse_graphml(FILE_PATH)
print(f"Full graph  — nodes: {G_full.number_of_nodes():,}  edges: {G_full.number_of_edges():,}")

# Extract edge features from the full graph for later use
edge_df = extract_edge_features(G_full)
print("\nEdge feature columns:", edge_df.columns.tolist())
print(edge_df.describe())

## 4. Subsample to a manageable size

The full graph has 41k nodes. Using a one-hot or degree feature matrix on that is expensive,
so we subsample to ~2000 nodes / 8000 edges for development.

In [ ]:


def get_dense_subgraph(G_full, target_nodes=2000):
    # 1. Pick a random starting node that actually has connections
    random.seed(SEED)
    start_node = random.choice([n for n, d in G_full.degree() if d > 5])
    
    # 2. Use BFS to find the closest 2000 nodes
    nodes = {start_node}
    queue = [start_node]
    
    while len(nodes) < target_nodes and queue:
        current = queue.pop(0)
        for neighbor in G_full.neighbors(current):
            if neighbor not in nodes:
                nodes.add(neighbor)
                queue.append(neighbor)
                if len(nodes) >= target_nodes:
                    break
                    
    # 3. Create the induced subgraph
    return G_full.subgraph(nodes).copy()

# Update your Section 4 with this:
G_sub = get_dense_subgraph(G_full, target_nodes=3000)
G_int = nx.convert_node_labels_to_integers(G_sub)

print(f"Subgraph — nodes: {G_sub.number_of_nodes():,}  edges: {G_sub.number_of_edges():,}")

print(f"Relabelled  — nodes: {G_int.number_of_nodes():,}  edges: {G_int.number_of_edges():,}")

## visualize graph

In [ ]:


# def visualize_split(G_int, train_edges, val_edges, test_edges):
#     plt.figure(figsize=(10, 7))
#     pos = nx.spring_layout(G_int, seed=42) # Consistent layout
    
#     # Draw nodes
#     nx.draw_networkx_nodes(G_int, pos, node_size=20, node_color='lightgrey', alpha=0.8)
    
#     # Draw edges with different colors
#     nx.draw_networkx_edges(G_int, pos, edgelist=train_edges, edge_color='blue', label='Train', width=1, alpha=0.5)
#     nx.draw_networkx_edges(G_int, pos, edgelist=val_edges, edge_color='orange', label='Val', width=1.5)
#     nx.draw_networkx_edges(G_int, pos, edgelist=test_edges, edge_color='red', label='Test', width=1.5)
    
#     plt.title("Graph Edge Split Visualization")
#     plt.legend()
#     plt.show()

# # Run the visualization
# visualize_split(G_int, train_edges, val_edges, test_edges)

## 5. Edge train / val / test split

In [ ]:
def preview_graph_edges(G, name, n=5):
    edges = list(G.edges(data=True))[:n]
    print(f"--- {name} (first {n} edges) ---")
    for u, v, edata in edges:
        # MultiGraph: edata is {0: {attrs}, 1: {attrs}, ...}
        inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
        label = inner.get('ActivityLabel', 'N/A')
        print(f"  ({u}, {v}) → ActivityLabel: {label}")


# ── Separate edges by label ───────────────────────────────────────────────────
normal_edges = []
attack_edges = []

for u, v, edata in G_int.edges(data=True):
    inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
    label = inner.get('ActivityLabel', 0)
    if label == 0:
        normal_edges.append((u, v))
    else:
        attack_edges.append((u, v))

print(f"Normal edges: {len(normal_edges):,}")
print(f"Attack edges: {len(attack_edges):,}")

# ── Split normal edges into train / val / test ────────────────────────────────
random.seed(SEED)
random.shuffle(normal_edges)

n             = len(normal_edges)
train_split   = int(0.70 * n)
val_split     = int(0.85 * n)

train_edges       = normal_edges[:train_split]           # 70% normal — training only
val_edges         = normal_edges[train_split:val_split]  # 15% normal — hyperparameter tuning
test_normal_edges = normal_edges[val_split:]             # 15% normal — held-out for test
test_attack_edges = attack_edges                         # ALL attacks — test only

print(f"Train edges  (normal only): {len(train_edges):,}")
print(f"Val edges    (normal only): {len(val_edges):,}")
print(f"Test normal  (normal only): {len(test_normal_edges):,}")
print(f"Test attack  (attack only): {len(test_attack_edges):,}")
print(f"Test anomaly ratio:         {len(test_attack_edges) / (len(test_normal_edges) + len(test_attack_edges)):.2%}")

# ── Build graphs ──────────────────────────────────────────────────────────────
G_train = nx.MultiGraph()
G_train.add_nodes_from(G_int.nodes(data=True))
G_train.add_edges_from((u, v, G_int[u][v]) for u, v in train_edges)

G_val = nx.MultiGraph()
G_val.add_nodes_from(G_int.nodes(data=True))
G_val.add_edges_from((u, v, G_int[u][v]) for u, v in val_edges)

print(f"G_train — nodes: {G_train.number_of_nodes():,}  edges: {G_train.number_of_edges():,}")
preview_graph_edges(G_train, "G_train")
print(f"G_val   — nodes: {G_val.number_of_nodes():,}  edges: {G_val.number_of_edges():,}")
preview_graph_edges(G_val,   "G_val")

## 6. Negative edge sampling

For each split we generate an equal number of fake (non-existent) edges.
These act as the negative class for the binary classification problem.

In [ ]:
# node2idx: node_id -> integer index
node2idx = {n: i for i, n in enumerate(G_train.nodes())}
def to_idx(edges):
    """Convert a list of (u, v) node-ID pairs to integer index pairs.
    Drops any edge whose endpoints are not in node2idx (e.g. isolated val/test nodes).
    """
    return [
        (node2idx[u], node2idx[v])
        for u, v in edges
        if u in node2idx and v in node2idx
    ]

EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']


NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)  # 7

In [ ]:
def generate_negative_edges(G, num_samples, seed=42):
    """Sample edges that do NOT exist in G."""
    random.seed(seed)
    neg = []
    nodes = list(G.nodes())
    existing = set(G.edges()) | {(v, u) for u, v in G.edges()}
    while len(neg) < num_samples:
        u, v = random.sample(nodes, 2)
        if (u, v) not in existing:
            neg.append((u, v))
    return neg


# Generate negative edges from G_train (structural non-edges)
neg_train = generate_negative_edges(G_train, len(train_edges), seed=42)
neg_val   = generate_negative_edges(G_train, len(val_edges),   seed=43)

# Convert to integer indices — drop any pairs outside node2idx
neg_train_idx = to_idx(neg_train)
neg_val_idx   = to_idx(neg_val)

# Negative edges have no traffic features — zeros is meaningful signal here
neg_train_attr = torch.zeros(len(neg_train_idx), NUM_EDGE_FEATS)
neg_val_attr   = torch.zeros(len(neg_val_idx),   NUM_EDGE_FEATS)

print(f"Negative train: {len(neg_train_idx):,}")
print(f"Negative val:   {len(neg_val_idx):,}")

## 7. Build node index map and integer edge lists

`node2idx` maps each node ID to a row index in the feature matrix.
All edge lists must use these integer indices for PyG.

In [ ]:





train_pos_idx = to_idx(train_edges)
val_pos_idx   = to_idx(val_edges)

test_normal_idx = to_idx(test_normal_edges)
test_attack_idx = to_idx(test_attack_edges)


print(f"train_pos_idx: {len(train_pos_idx):,}")
print(f"val_pos_idx:   {len(val_pos_idx):,}")
print(f"test_normal_idx:  {len(test_normal_idx):,}")
print(f"test_attack_idx:  {len(test_attack_idx):,}")



## 8. Build PyG Data object with edge features

Node features: normalised degree (1 scalar per node — cheap and informative).
Edge features: the 7 traffic columns from the dataset.

In [ ]:
EDGE_FEAT_COLS = ['TotPkts', 'TotBytes', 'SrcBytes', 'Dur',
                  'Proto_encoded', 'Dir_encoded', 'State_encoded']

NUM_EDGE_FEATS = len(EDGE_FEAT_COLS)

# ── Node features ─────────────────────────────────────────────────────────────
node_feat_dict = {n: [] for n in G_train.nodes()}
for u, v, edata in G_train.edges(data=True):
    for key in edata:
        inner = edata[key]
        feats = [float(inner.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
        node_feat_dict[u].append(feats)
        node_feat_dict[v].append(feats)

node_feats = []
for n in G_train.nodes():
    edge_feats = node_feat_dict[n]
    if edge_feats:
        arr = np.array(edge_feats)
        agg = np.concatenate([
            np.mean(arr, axis=0),
            np.std(arr, axis=0),
            np.min(arr, axis=0),
            np.max(arr, axis=0),
        ])
    else:
        agg = np.zeros(NUM_EDGE_FEATS * 4)
    node_feats.append(agg)

degrees = np.array([G_train.degree(n) for n in G_train.nodes()], dtype=float).reshape(-1, 1)
node_feats = np.array(node_feats)
node_feats = np.concatenate([node_feats, degrees], axis=1)  # → [N, 29]

deg_col = node_feats[:, -1]
node_feats[:, -1] = (deg_col - deg_col.mean()) / (deg_col.std() + 1e-9)

x = torch.tensor(node_feats, dtype=torch.float)

print(f"Node feature matrix: {x.shape}")
print(f"Sample row (node 0): {x[0]}")
print(f"Feature means: {x.mean(dim=0)}")
print(f"Feature stds:  {x.std(dim=0)}")

# ── Edge index and edge_attr for message passing (G_train only) ───────────────
train_edge_list = list(G_train.edges())

edge_index = torch.tensor(
    [[node2idx[u], node2idx[v]] for u, v in train_edge_list],
    dtype=torch.long
).t().contiguous()

edge_attr = []
for u, v in train_edge_list:
    raw_data = G_train[u][v][0]
    feats = [float(raw_data.get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    edge_attr.append(feats)

edge_attr = torch.tensor(edge_attr, dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

print("Data object:")
print(f"  x:          {data.x.shape}")
print(f"  edge_index: {data.edge_index.shape}")
print(f"  edge_attr:  {data.edge_attr.shape}")

# ── Edge attrs for scoring — all use G_int as source of truth ─────────────────
idx2node = {i: n for n, i in node2idx.items()}

train_edges_attr = torch.tensor([
    [float(G_int[idx2node[u]][idx2node[v]][0].get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    for u, v in train_pos_idx
], dtype=torch.float)

val_edges_attr = torch.tensor([
    [float(G_int[idx2node[u]][idx2node[v]][0].get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    for u, v in val_pos_idx
], dtype=torch.float)

all_test_edges = test_normal_idx + test_attack_idx
test_edges_attr = torch.tensor([
    [float(G_int[idx2node[u]][idx2node[v]][0].get(col, 0.0) or 0.0) for col in EDGE_FEAT_COLS]
    for u, v in all_test_edges
], dtype=torch.float)

print(f"  train_edges_attr: {train_edges_attr.shape}")
print(f"  val_edges_attr:   {val_edges_attr.shape}")
print(f"  test_edges_attr:  {test_edges_attr.shape}")


## 9. NNConv model

NNConv uses edge features to compute a per-edge weight matrix that transforms
the source node's features before aggregating into the target node.
This means every edge contributes differently to the message passing,
which is exactly what we want when traffic features carry the signal.

In [ ]:
HIDDEN = 128


class GNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden):
        super().__init__()

        # Conv 1: node_in -> hidden
        # edge_nn output must equal node_in * hidden (produces the weight matrix)
        self.conv1 = NNConv(
            in_channels=node_in,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, node_in * hidden),
                nn.ReLU()
            )
        )
        self.bn1 = nn.BatchNorm1d(hidden)

        # Conv 2: hidden -> hidden
        self.conv2 = NNConv(
            in_channels=hidden,
            out_channels=hidden,
            nn=nn.Sequential(
                nn.Linear(edge_in, hidden * hidden),
                nn.ReLU()
            )
        )
        self.bn2 = nn.BatchNorm1d(hidden)

        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden + edge_in, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 1)
        
        )

    def encode(self, x, edge_index, edge_attr):
        """Run message passing to produce node embeddings."""
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = self.bn2(self.conv2(x, edge_index, edge_attr))
        return x

    def decode(self, z, edges, edges_attr):
        u = torch.tensor([e[0] for e in edges], dtype=torch.long)
        v = torch.tensor([e[1] for e in edges], dtype=torch.long)
        # Concat node embeddings + edge features and pass through MLP
        combined = torch.cat([z[u], z[v], edges_attr], dim=1)
        return self.mlp(combined).squeeze()

    def forward(self, x, edge_index, edge_attr, edges):
        z = self.encode(x, edge_index, edge_attr)
        return self.decode(z, edges, edge_attr)


model = GNN(
    node_in=x.shape[1],
    edge_in=NUM_EDGE_FEATS,
    hidden=HIDDEN
)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#data diagnostic

# ── DIAGNOSTIC: examine attack edge properties before model changes ────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Pull raw feature vectors for normal vs attack edges ───────────────────────
def get_edge_features(G_int, edge_list, feat_cols):
    rows = []
    for u, v in edge_list:
        # MultiGraph: grab first parallel edge
        edata = G_int[u][v]
        inner = edata[0] if isinstance(edata, dict) and 0 in edata else edata
        row = {col: float(inner.get(col, 0.0) or 0.0) for col in feat_cols}
        rows.append(row)
    return pd.DataFrame(rows)

normal_df = get_edge_features(G_int, test_normal_edges, EDGE_FEAT_COLS)
attack_df = get_edge_features(G_int, test_attack_edges, EDGE_FEAT_COLS)

print("=== Normal edge feature stats ===")
print(normal_df.describe().round(4))
print("\n=== Attack edge feature stats ===")
print(attack_df.describe().round(4))

# ── Per-feature mean comparison ───────────────────────────────────────────────
comparison = pd.DataFrame({
    'normal_mean': normal_df.mean(),
    'attack_mean': attack_df.mean(),
    'normal_std':  normal_df.std(),
    'attack_std':  attack_df.std(),
})
comparison['mean_diff'] = (comparison['attack_mean'] - comparison['normal_mean']).abs()
comparison['separable'] = comparison['mean_diff'] > comparison['normal_std']
print("\n=== Feature separability ===")
print(comparison.round(4))

# ── Degree analysis: are attack nodes high- or low-degree? ────────────────────
attack_nodes = set([u for u, v in test_attack_edges] + [v for u, v in test_attack_edges])
normal_nodes  = set([u for u, v in test_normal_edges] + [v for u, v in test_normal_edges])

attack_degrees = [G_int.degree(n) for n in attack_nodes]
normal_degrees = [G_int.degree(n) for n in normal_nodes]

print(f"\nAttack node degree  — mean: {np.mean(attack_degrees):.2f}  "
      f"median: {np.median(attack_degrees):.2f}  max: {max(attack_degrees)}")
print(f"Normal node degree  — mean: {np.mean(normal_degrees):.2f}  "
      f"median: {np.median(normal_degrees):.2f}  max: {max(normal_degrees)}")

# ── Overlap: how many attack nodes also appear in normal edges? ────────────────
overlap = attack_nodes & normal_nodes
print(f"\nAttack nodes that also appear in normal edges: "
      f"{len(overlap)}/{len(attack_nodes)} ({len(overlap)/len(attack_nodes):.1%})")

# ── Feature distribution plots ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, col in enumerate(EDGE_FEAT_COLS):
    axes[i].hist(normal_df[col], bins=40, alpha=0.6, label='Normal', color='blue', density=True)
    axes[i].hist(attack_df[col], bins=40, alpha=0.6, label='Attack', color='red', density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)
axes[-1].axis('off')
plt.suptitle('Feature Distributions: Normal vs Attack')
plt.tight_layout()
plt.show()

## 10. Training

In [ ]:

def evaluate(model, data, val_normal_idx, val_edges_attr, neg_val_idx, neg_val_attr):
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)
        pos_scores = torch.sigmoid(model.decode(z, val_normal_idx, val_edges_attr))
        neg_scores = torch.sigmoid(model.decode(z, neg_val_idx, neg_val_attr))

    all_scores    = torch.cat([pos_scores, neg_scores]).cpu().numpy()
    anomaly_scores = 1 - all_scores
    y_true        = [0] * len(val_normal_idx) + [1] * len(neg_val_idx)
    val_auc       = roc_auc_score(y_true, anomaly_scores)

    gap = pos_scores.mean().item() - neg_scores.mean().item()
    print(f"  Val normal score (mean): {pos_scores.mean():.4f} "
          f"| Val negative score (mean): {neg_scores.mean():.4f} "
          f"| Gap: {gap:.4f} | Val AUROC: {val_auc:.4f}")

    # ── CHANGED: return AUROC as the early stopping signal ────────────────────
    return val_auc
    # ──────────────────────────────────────────────────────────────────────────


def evaluate_test(model, data, test_normal_idx, test_attack_idx, test_edges_attr):
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index, data.edge_attr)
        all_edges = test_normal_idx + test_attack_idx
        # Apply sigmoid here to get probabilities
        scores = torch.sigmoid(
            model.decode(z, all_edges, test_edges_attr)
        ).cpu().numpy()

    # Normal edges score high, attack edges score low
    # So anomaly score = 1 - score
    anomaly_scores = 1 - scores
    y_true = [0] * len(test_normal_idx) + [1] * len(test_attack_idx)

    auc  = roc_auc_score(y_true, anomaly_scores)
    ap   = average_precision_score(y_true, anomaly_scores)

    # Find optimal threshold via val set ideally, but P-R curve as fallback
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true, anomaly_scores)
    f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-9)
    best_thresh = thresholds[np.argmax(f1_vals)]

    y_pred = (anomaly_scores >= best_thresh).astype(int)

    # Threshold-dependent
    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    f1        = f1_score(y_true, y_pred)
    f2        = fbeta_score(y_true, y_pred, beta=2)  # recall-weighted

    # FPR at 90% recall
    target_recall = 0.90
    valid = recall_vals >= target_recall
    fpr_at_90_recall = 1 - precision_vals[valid][-1] if valid.any() else float('nan')

    print(f"Test AUROC:            {auc:.4f}")
    print(f"Test AUPRC:            {ap:.4f}")
    print(f"Best threshold:        {best_thresh:.4f}")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")
    print(f"F1:                    {f1:.4f}")
    print(f"F2 (recall-weighted):  {f2:.4f}")
    print(f"FPR at 90% Recall:     {fpr_at_90_recall:.4f}")

    return auc, ap, f1, f2


In [ ]:
# ── CHANGED: renamed BEST_GAP to BEST_AUC -- tracking AUROC not gap ───────────
BEST_AUC = 0.0
# ──────────────────────────────────────────────────────────────────────────────
PATIENCE = 15           # CHANGED: reduced from 30 per last recommendation
NO_IMPROVE_COUNT = 0
BEST_MODEL_PATH = 'best_model.pt'
EPOCHS = 300

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    z = model.encode(data.x, data.edge_index, data.edge_attr)

    pos_labels = torch.ones(len(train_pos_idx))
    neg_labels = torch.zeros(len(neg_train_idx))

    pos_scores = model.decode(z, train_pos_idx, train_edges_attr)
    neg_scores = model.decode(z, neg_train_idx, neg_train_attr)

    pos_loss = criterion(pos_scores, pos_labels)
    neg_loss = criterion(neg_scores, neg_labels)
    loss = pos_loss + neg_loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if epoch % 10 == 0:
        # ── CHANGED: evaluate() now returns val AUROC -- use that return value --
        # removed the redundant manual gap computation block that was running
        # a second forward pass. evaluate() handles everything internally.
        val_auc = evaluate(model, data, val_pos_idx, val_edges_attr,
                           neg_val_idx, neg_val_attr)
        # ──────────────────────────────────────────────────────────────────────

        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} "
              f"| Pos loss: {pos_loss.item():.4f} "
              f"| Neg loss: {neg_loss.item():.4f}")

        # ── CHANGED: compare AUROC not gap ────────────────────────────────────
        if val_auc > BEST_AUC:
            BEST_AUC = val_auc
            NO_IMPROVE_COUNT = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  >> New best val AUROC: {BEST_AUC:.4f} — checkpoint saved")
        else:
            NO_IMPROVE_COUNT += 1
            print(f"  >> No improvement for {NO_IMPROVE_COUNT}/{PATIENCE} evals "
                  f"(best AUROC: {BEST_AUC:.4f})")
            if NO_IMPROVE_COUNT >= PATIENCE:
                print(f"  >> Early stopping triggered at epoch {epoch}.")
                break
        # ──────────────────────────────────────────────────────────────────────

        # ── CHANGED: scheduler now steps on val_auc, which is defined above ───
        scheduler.step(val_auc)
        # ──────────────────────────────────────────────────────────────────────

print(f"\nLoading best model (AUROC: {BEST_AUC:.4f}) from '{BEST_MODEL_PATH}'")
model.load_state_dict(torch.load(BEST_MODEL_PATH))

## 11. Final test evaluation

In [ ]:
auc, ap, f1, f2 = evaluate_test(model, data, test_normal_idx, test_attack_idx, test_edges_attr)




In [ ]:
model.eval()
with torch.no_grad():
    z = model.encode(data.x, data.edge_index, data.edge_attr)
    
    normal_scores = model.decode(z, test_normal_idx, 
                                  test_edges_attr[:len(test_normal_idx)]).cpu().numpy()
    attack_scores = model.decode(z, test_attack_idx,
                                  test_edges_attr[len(test_normal_idx):]).cpu().numpy()

print(f"Normal edges  — mean: {normal_scores.mean():.4f}  std: {normal_scores.std():.4f}  "
      f"min: {normal_scores.min():.4f}  max: {normal_scores.max():.4f}")
print(f"Attack edges  — mean: {attack_scores.mean():.4f}  std: {attack_scores.std():.4f}  "
      f"min: {attack_scores.min():.4f}  max: {attack_scores.max():.4f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', color='blue')
plt.hist(attack_scores, bins=50, alpha=0.6, label='Attack', color='red')
plt.xlabel('Model Score')
plt.ylabel('Count')
plt.title('Score Distribution: Normal vs Attack')
plt.legend()
plt.show()